# Step 1.4b: What do the SpeechArenaBench labels mean?

**Run on the server** (kernel = the `hinglishtts` venv). **Read-only.** It reads only our own filtered copy, `datasets/speecharena_hi_AB/` (the 4,444 pairs of the allowed systems), and never loads the audio columns.

Questions it answers (each one affects the loader, Step 1.5):
1. What values does `preference_model` take, and does it include ties ("both good", "both bad")?
2. Is each row one rater's vote, or an aggregated pair?
3. Does position matter? In SpeechJudge, clip B won more often than A; is the same true here?
4. What is inside `fine_grained_eval`?
5. Raters: how many, how many votes each, any gender or language fields to use?

Everything shown is also saved to `analysis/step1_4b/output.log`, the loaded table is cached as parquet, and a `summary.json` is saved at the end.

In [ ]:
import collections
import json
import sys
from pathlib import Path

import pandas as pd
import pyarrow
import pyarrow.parquet

ROOT = Path.home() / "projects" / "hinglish-cs-tts"
DS = ROOT / "datasets"

# everything shown with log() is also saved to output.log, so no result lives only in kernel memory
OUTDIR = ROOT / "analysis" / "step1_4b"
OUTDIR.mkdir(parents=True, exist_ok=True)


def log(*parts):
    """Print a line and append it to output.log."""
    line = " ".join([str(part) for part in parts])
    print(line)
    with open(OUTDIR / "output.log", "a") as log_file:
        log_file.write(line + "\n")


def save_json(name, data):
    """Save data as JSON under OUTDIR, atomically (temp file, then rename)."""
    tmp = OUTDIR / (name + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str))
    tmp.replace(OUTDIR / name)


log("python", sys.version.split()[0], "| pandas", pd.__version__, "| pyarrow", pyarrow.__version__)
log("outputs and results are saved under", OUTDIR)

In [ ]:
# 1. Load the metadata columns of our 4,444 kept pairs (audio columns are never read). Cached on disk as parquet.
SAB = DS / "speecharena_hi_AB"
shard_files = sorted(SAB.glob("*.parquet"))
log(len(shard_files), "parquet files in", SAB.name)
names = pyarrow.parquet.ParquetFile(shard_files[0]).schema_arrow.names
log("columns:", names)
cols = [c for c in names if c not in ["audio_a", "audio_b"]]
cache = OUTDIR / "sab_meta.parquet"
if cache.exists():
    df = pd.read_parquet(cache)
    log("loaded from the disk cache:", len(df), "rows")
else:
    tables = []
    for path in shard_files:
        tables.append(pyarrow.parquet.read_table(path, columns=cols))
    df = pyarrow.concat_tables(tables).to_pandas()
    tmp = cache.with_suffix(".tmp")
    df.to_parquet(tmp)
    tmp.replace(cache)
    log("read", len(df), "rows and saved", cache.name)
if len(df) == 4444:
    log("expected 4444 rows -> OK")
else:
    log("expected 4444 rows -> DIFFERENT, investigate")

In [ ]:
# 2. preference_model: which values exist, and how they relate to mixed vs single-script sentences
def has_devanagari(text):
    """True if the text has a Devanagari character (Unicode block U+0900 to U+097F)."""
    for ch in text:
        if "ऀ" <= ch <= "ॿ":
            return True
    return False


def has_latin(text):
    """True if the text has an English letter A-Z or a-z."""
    for ch in text:
        if ch.isascii() and ch.isalpha():
            return True
    return False


def kind(sentence):
    """mixed (Devanagari and Latin letters), latin_only, dev_only or other."""
    sentence = sentence or ""
    devanagari = has_devanagari(sentence)
    latin = has_latin(sentence)
    if devanagari and latin:
        return "mixed"
    if latin:
        return "latin_only"
    if devanagari:
        return "dev_only"
    return "other"


df["kind"] = df["sentence"].map(kind)
log("preference_model values:")
log(df["preference_model"].value_counts(dropna=False).to_string())
log("\nby sentence kind:")
log(pd.crosstab(df["preference_model"], df["kind"]).to_string())
log("\nmixed rows (expected 1159):", int((df["kind"] == "mixed").sum()))

In [ ]:
# 3. Are the values system names or A/B labels? Does position matter (does model_b win more often than model_a)?
values = set(df["preference_model"].dropna().unique())
systems = set(df["model_a"]) | set(df["model_b"])
log("values that are system names:", sorted(values & systems))
log("other values (labels such as ties):", sorted(values - systems))
win_a = int((df["preference_model"] == df["model_a"]).sum())
win_b = int((df["preference_model"] == df["model_b"]).sum())
log(f"winner is model_a: {win_a} | winner is model_b: {win_b} | neither (tie / both / other): {len(df) - win_a - win_b}")
if win_a + win_b:
    log(f"share of decided votes where model_a (first position) wins: {win_a / (win_a + win_b):.3f}  (0.5 would mean no position bias)")
log("\nhow often each system is in position A (rows) against position B (columns):")
log(pd.crosstab(df["model_a"], df["model_b"]).to_string())

In [ ]:
# 4. Is each row one rater's vote? Votes per item, raters, gender, language
key = df["academic_prompt_id"].astype(str) + "|" + df["model_a"] + "|" + df["model_b"]
log("rows:", len(df), "| distinct (sentence, system A, system B) items:", key.nunique())
log("votes per item:", key.value_counts().value_counts().sort_index().to_dict())
log("distinct sentences:", df["academic_prompt_id"].nunique())
log("unique raters:", df["user_id"].nunique())
log("votes per rater (min / median / max):", df.groupby("user_id").size().agg(["min", "median", "max"]).to_dict())
log("gender:", df["gender"].value_counts(dropna=False).to_dict())
log("language:", df["language"].value_counts(dropna=False).to_dict())

In [ ]:
# 5. What is inside fine_grained_eval?
def parse(x):
    """JSON text becomes a Python object; anything else is returned unchanged."""
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x


fg = df["fine_grained_eval"].dropna().map(parse)
log("missing fine_grained_eval:", int(df["fine_grained_eval"].isna().sum()), "of", len(df))
for value in fg.iloc[:3]:
    log(type(value).__name__, json.dumps(value, ensure_ascii=False, default=str)[:700])
keys = None
if len(fg) and isinstance(fg.iloc[0], dict):
    keys = collections.Counter()
    for value in fg:
        if isinstance(value, dict):
            for k in value:
                keys[k] += 1
    log("\nkeys:", dict(keys))
    for k in list(keys)[:10]:
        texts = []
        for value in fg:
            if isinstance(value, dict):
                texts.append(str(value.get(k)))
            else:
                texts.append(None)
        log(" ", k, "->", pd.Series(texts).value_counts().head(6).to_dict())

## Summary (saved to `analysis/step1_4b/summary.json`)

In [ ]:
share_first = None
if win_a + win_b:
    share_first = round(win_a / (win_a + win_b), 4)
fine_grained_keys = None
if keys is not None:
    fine_grained_keys = dict(keys)
summary = {
    "rows": len(df),
    "mixed_rows": int((df["kind"] == "mixed").sum()),
    "preference_model_values": df["preference_model"].value_counts(dropna=False).to_dict(),
    "winner_is_model_a": win_a,
    "winner_is_model_b": win_b,
    "neither": len(df) - win_a - win_b,
    "share_first_position_wins": share_first,
    "distinct_items": int(key.nunique()),
    "votes_per_item": key.value_counts().value_counts().sort_index().to_dict(),
    "distinct_sentences": int(df["academic_prompt_id"].nunique()),
    "unique_raters": int(df["user_id"].nunique()),
    "gender": df["gender"].value_counts(dropna=False).to_dict(),
    "language": df["language"].value_counts(dropna=False).to_dict(),
    "fine_grained_missing": int(df["fine_grained_eval"].isna().sum()),
    "fine_grained_keys": fine_grained_keys,
}
save_json("summary.json", summary)
log("=== SUMMARY (also saved as summary.json) ===")
log(json.dumps(summary, indent=1, ensure_ascii=False, default=str)[:3000])